# Parallel Batch Computation

`compute_parallel()` runs multiple isochrone requests concurrently using
worker processes (or threads). It is the right tool for:

- Proof-of-principle runs over a list of cities
- Global grid runs on a SLURM cluster
- Any workload where sequential `compute_batch()` is too slow

**What's covered**
1. Basic parallel run over a city list
2. `router_factory` — why each worker needs its own router
3. `output_dir` — write results to disk as each one completes
4. `skip_existing` — resume an interrupted run without recomputing
5. `on_error` — fault-tolerant runs that continue past failures
6. `use_threads=True` — when to prefer threads over processes (ORS)
7. Scaling: estimating wall-clock time for large runs


## Setup

In [ ]:
import tempfile
import time
import warnings
from pathlib import Path

from idd_travel_time.models import TravelTimeRequest, TransitProfile
from idd_travel_time.constants import TransportMode, TimeContext
from idd_travel_time.isochrone import compute_parallel
from idd_travel_time.routers import OsmnxRouter
from idd_travel_time.io import GraphCache, result_id, read_result

RESULTS_DIR = Path(tempfile.mkdtemp()) / "batch_results"
CACHE_DIR   = Path(tempfile.mkdtemp()) / "graph_cache"


## 1 · Define a city list

In [ ]:
# Sub-Saharan African cities — walking isochrone from city centre
cities = [
    ("Nairobi, Kenya",         -1.2864, 36.8172, "KE"),
    ("Kampala, Uganda",         0.3163, 32.5822, "UG"),
    ("Kigali, Rwanda",         -1.9441, 30.0619, "RW"),
    ("Dar es Salaam, Tanzania", -6.7924, 39.2083, "TZ"),
]

requests = [
    TravelTimeRequest(
        origin_lat=lat, origin_lon=lon,
        mode=TransportMode.WALKING,
        duration_minutes=20,
    )
    for _, lat, lon, _ in cities
]

print(f"{len(requests)} requests queued")
for (name, *_), req in zip(cities, requests):
    print(f"  {name}")


## 2 · `router_factory` — one router per worker

`compute_parallel` spawns worker processes. Each worker must instantiate
its own router — routers are not safely shareable across processes.
`router_factory` is a zero-argument callable called once per worker.


In [ ]:
cache = GraphCache(CACHE_DIR)

# router_factory is called once in each worker process
def make_router():
    return OsmnxRouter(graph_cache=cache)

t0 = time.monotonic()
results = compute_parallel(
    requests,
    router_factory=make_router,
    max_workers=2,           # 2 worker processes in parallel
    output_dir=RESULTS_DIR,  # write to disk as each completes
)
elapsed = time.monotonic() - t0

print(f"Completed {len(results)} isochrones in {elapsed:.1f}s")
for (name, *_), res in zip(cities, results):
    if res is not None:
        iso_utm = res.isochrone_gdf.to_crs(res.isochrone_gdf.estimate_utm_crs())
        area = iso_utm.geometry.area.iloc[0] / 1e6
        print(f"  ✓  {name:<35}  {area:.2f} km²")
    else:
        print(f"  ✗  {name}")


## 3 · `output_dir` — write-as-you-go

Each result is written to disk immediately after it completes, before the
next worker starts. This means a crash or timeout loses at most one
in-flight result, not the whole batch.


In [ ]:
# Inspect what's in the results directory
print("On-disk results:")
for d in sorted(RESULTS_DIR.iterdir()):
    if d.is_dir():
        loaded = read_result(d)
        print(f"  {d.name[:50]}")
        print(f"    origin: ({loaded.request.origin_lat:.4f}, {loaded.request.origin_lon:.4f})")


## 4 · `skip_existing=True` — resume interrupted runs

Re-running with `skip_existing=True` (the default) loads completed results
from disk and only recomputes missing ones. Safe to run multiple times.


In [ ]:
# Simulate a new session — results are already on disk
t0 = time.monotonic()
results_resumed = compute_parallel(
    requests,
    router_factory=make_router,
    max_workers=2,
    output_dir=RESULTS_DIR,
    skip_existing=True,   # default — skips already-computed results
)
elapsed_resume = time.monotonic() - t0

print(f"Resume run completed in {elapsed_resume:.2f}s (loaded from disk, no routing)")
print(f"All results present: {all(r is not None for r in results_resumed)}")


## 5 · `on_error` — fault-tolerant batch runs

For large batches, some locations will fail (ambiguous geocoding, Overpass
timeout, unreachable area). `on_error="warn"` continues past failures and
inserts `None` in the results list.


In [ ]:
# Mix valid and invalid requests
mixed_requests = requests + [
    TravelTimeRequest(
        origin_lat=0.0, origin_lon=0.0,   # middle of ocean — will fail or return empty
        mode=TransportMode.WALKING,
        duration_minutes=20,
    )
]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    mixed_results = compute_parallel(
        mixed_requests,
        router_factory=make_router,
        max_workers=2,
        on_error="warn",   # continue past failures
    )

successes = sum(1 for r in mixed_results if r is not None)
failures  = sum(1 for r in mixed_results if r is None)
print(f"Successes: {successes}  Failures: {failures}")
if caught:
    print(f"Warnings: {len(caught)}")
    for w in caught:
        print(f"  {w.message}")


## 6 · `use_threads=True` — for ORS / I/O-bound backends

`ProcessPoolExecutor` (the default) bypasses the GIL and is right for
CPU-bound routing (osmnx graph traversal). For I/O-bound backends like
ORS, threads are more efficient because most time is spent waiting for
HTTP responses.


In [ ]:
# ORS example (requires ORS_API_KEY to be set)
import os

if os.environ.get("ORS_API_KEY"):
    from idd_travel_time.routers import OpenRouteServiceRouter

    ors_results = compute_parallel(
        requests[:2],
        router_factory=OpenRouteServiceRouter,
        max_workers=4,
        use_threads=True,      # I/O-bound — threads are better
    )
    print(f"ORS results: {[r.router_name for r in ors_results if r]}")
else:
    print("ORS_API_KEY not set — skipping ORS example.")
    print("Set ORS_API_KEY and re-run to see the thread-based parallel ORS workflow.")


## 7 · Estimating wall-clock time for large runs

```
T_wall ≈ (N_requests / max_workers) × T_per_request

where:
  T_per_request ≈ graph_download_time + routing_time
                ≈ 20s (uncached) or 1s (cached) for a 20-min walking request
```


In [ ]:
import math

n_cities = 500        # e.g. all SSA cities > 100k population
workers  = 8
t_uncached_s = 25     # typical uncached Overpass download
t_cached_s   = 1      # cached graph read + routing

wall_uncached = (n_cities / workers) * t_uncached_s / 60
wall_cached   = (n_cities / workers) * t_cached_s   / 60

print(f"Estimating wall-clock time for {n_cities} cities on {workers} workers:")
print(f"  First run (no graph cache):  {wall_uncached:.0f} min  ({wall_uncached/60:.1f} hr)")
print(f"  Re-run    (graph cache hit): {wall_cached:.0f} min")
print()
print("→ Pre-warm the graph cache on a small representative run before scaling.")
